In [ ]:
"""
Full AC solver matching the verifier's core.py exactly.
"""

import json
from collections import deque
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------- Core semantics (from verifier/core.py) ----------
def free_reduce(word):
    out = []
    for a in word:
        if out and out[-1] == -a:
            out.pop()
        else:
            out.append(a)
    return tuple(out)

def invert(word):
    return tuple(-a for a in reversed(word))

GENS = (1, -1, 2, -2)

def apply_move(state, m):
    r0, r1 = state
    if m == 0:  return (invert(r0), r1)
    if m == 1:  return (r0, invert(r1))
    if m == 2:  return (free_reduce(r0 + r1), r1)
    if m == 3:  return (free_reduce(r0 + invert(r1)), r1)
    if m == 4:  return (r0, free_reduce(r1 + r0))
    if m == 5:  return (r0, free_reduce(r1 + invert(r0)))
    if 6 <= m <= 9:
        g = GENS[m - 6]
        return (free_reduce((g,) + r0 + (-g,)), r1)
    if 10 <= m <= 13:
        g = GENS[m - 10]
        return (r0, free_reduce((g,) + r1 + (-g,)))
    raise ValueError("bad move id: %r" % (m,))

# ---------- Parser: strings -> integers ----------
CHAR_TO_INT = {"x": 1, "X": -1, "y": 2, "Y": -2}

def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if chunk:
            w = encode(chunk.replace("= 1", "").strip())
            relators.append(tuple(CHAR_TO_INT[c] for c in w))
    return gens_part.strip().split(), relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            _, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "initial": (rels[0], rels[1]),
            })
    return problems

# ---------- Goal ----------
TARGET = ((1,), (2,))     # exact ordered pair (x, y)

def state_key(state):
    return state

# ---------- Bidirectional BFS ----------
def bidirectional_solve(initial, max_depth=20):
    if initial == TARGET:
        return []

    fwd = deque([(initial, [])])
    bwd = deque([(TARGET, [])])
    fwd_seen = {initial: []}
    bwd_seen = {TARGET: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                if ns in bwd_seen:
                    return path + [m] + bwd_seen[ns]
                if ns not in fwd_seen:
                    fwd_seen[ns] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                if ns in fwd_seen:
                    return fwd_seen[ns] + [m] + path
                if ns not in bwd_seen:
                    bwd_seen[ns] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ---------- Sanity check: known solution ----------
print("=== Sanity check on ms-train-0001 ===")
state = ((-1, 2, 1, 2, -2), (1, 2))
known = [0, 9, 1, 2, 9, 0, 4, 1]
for m in known:
    state = apply_move(state, m)
print(f"after known moves: {state}")
print(f"target: {TARGET}")
print(f"MATCH: {state == TARGET}")

# ---------- Solve all 424 training problems ----------
def solve_one(args):
    cid, initial = args
    return cid, bidirectional_solve(initial, max_depth=20)

if __name__ == "__main__":
    AC_JSONL = Path(".") / "ac.jsonl"
    problems = load_jsonl(AC_JSONL)
    print(f"\nLoaded {len(problems)} problems.")

    tasks = [(p["challenge_id"], p["initial"]) for p in problems]
    results = {}

    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(solve_one, t) for t in tasks]
        done = 0
        for fut in as_completed(futures):
            cid, moves = fut.result()
            done += 1
            if moves is not None:
                results[cid] = moves
                print(f"[{done}/{len(tasks)}] {cid}: SOLVED ({len(moves)})")
            else:
                print(f"[{done}/{len(tasks)}] {cid}: NOT SOLVED")

    with open("practice_submission_int.txt", "w") as f:
        for cid, moves in results.items():
            f.write(f"{cid}: [{', '.join(str(m) for m in moves)}]\n")

    print(f"\nWrote {len(results)}/{len(problems)} solutions.")

=== Sanity check on ms-train-0001 ===
after known moves: ((1, 2, 2), (-2,))
target: ((1,), (2,))
MATCH: False

Loaded 424 problems.
[1/424] ms-train-0001: SOLVED (5)
[2/424] ms-train-0008: SOLVED (6)
[3/424] ms-train-0002: SOLVED (6)
[4/424] ms-train-0003: SOLVED (7)
[5/424] ms-train-0004: SOLVED (8)
[6/424] ms-train-0009: SOLVED (8)
[7/424] ms-train-0010: SOLVED (9)
[8/424] ms-train-0015: SOLVED (6)
[9/424] ms-train-0005: SOLVED (9)
[10/424] ms-train-0016: SOLVED (8)
[11/424] ms-train-0011: SOLVED (10)
[12/424] ms-train-0006: SOLVED (10)
[13/424] ms-train-0017: SOLVED (10)
[14/424] ms-train-0018: SOLVED (11)
[15/424] ms-train-0022: SOLVED (6)
[16/424] ms-train-0023: SOLVED (7)
[17/424] ms-train-0024: SOLVED (8)
[18/424] ms-train-0025: SOLVED (9)
[19/424] ms-train-0026: SOLVED (10)
[20/424] ms-train-0012: SOLVED (11)
[21/424] ms-train-0007: SOLVED (11)
[22/424] ms-train-0029: SOLVED (8)
[23/424] ms-train-0030: SOLVED (7)
[24/424] ms-train-0031: SOLVED (9)
[25/424] ms-train-0032: SOLVED